In [2]:
from datasets import load_dataset

ds = load_dataset("rpmon/fma-genre-classification")

c:\Users\tim7m\OneDrive\Desktop\gp5\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tim7m\OneDrive\Desktop\gp5\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tim7m\.cache\huggingface\hub\datasets--rpmon--fma-genre-classification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In o

In [10]:
df_train = ds['train'].to_pandas()
df_val = ds['validation'].to_pandas()

df_train.head()

,audio,genre,track_id,title,artist
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x041TIT2\x...,0,55183,Intro To Horror,Dylan Palme
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05\x1eTIT...,6,120184,Lavender Hip Mob,Lee Rosevere
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04aTIT2\x...,7,85816,Break It Now,Radio 421
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x06\x03TIT...,5,36277,"Restaurant Concert, Song 2",Demiran Ćerimović and His Orkestar
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04VTIT2\x...,7,56552,Celebration,Krebs


In [30]:
import io
import os
import librosa
import soundfile as sf

for df, out_dir in zip([df_train, df_val], ["data/audio_train", "data/audio_val"]):
    for _, row in df.iterrows():
        track_id = row['track_id']
        out_path = f"{out_dir}/{track_id}.wav"

        if os.path.exists(out_path):
            continue

        try:
            audio_bytes = row['audio']['bytes']
            y, sr = librosa.load(io.BytesIO(audio_bytes), sr=None)
            sf.write(out_path, y, sr)
        except Exception as e:
            print(track_id, e)
            continue

99134 Error opening <_io.BytesIO object at 0x00000271EE7DBF60>: File does not exist or is not a regular file (possibly a pipe?).
98565 Unspecified internal error.
108925 Error opening <_io.BytesIO object at 0x00000271EE7FAD40>: File does not exist or is not a regular file (possibly a pipe?).
133297 Error opening <_io.BytesIO object at 0x00000271EE7DBF60>: File does not exist or is not a regular file (possibly a pipe?).
97848 System error.
16820 System error.
107248 System error.
132589 System error.
40659 System error.
146481 System error.
97283 System error.
66637 System error.
88959 System error.
72149 System error.
121737 System error.
68354 System error.
694 System error.
119027 System error.
144214 System error.
143307 System error.
54703 System error.
47193 System error.
75377 System error.
99501 System error.
37131 System error.
91894 System error.
56639 System error.
7011 System error.
133436 System error.
128760 System error.
7393 System error.
38894 System error.
91443 System

In [50]:
import io
import os
import soundfile as sf
import numpy as np

for df, out_dir in zip(
    [df_train, df_val],
    ["data/audio_train", "data/audio_val"]
):

    os.makedirs(out_dir, exist_ok=True)

    for _, row in df.iterrows():

        track_id = row["track_id"]
        out_path = f"{out_dir}/{track_id}.wav"

        if os.path.exists(out_path):
            continue

        try:
            audio_bytes = row["audio"]["bytes"]

            y, sr = sf.read(io.BytesIO(audio_bytes))

            # stereo -> mono
            if len(y.shape) > 1:
                y = np.mean(y, axis=1)

            sf.write(out_path, y, sr)

        except Exception as e:
            print(track_id, e)

99134 Error opening <_io.BytesIO object at 0x00000271EE38B290>: File does not exist or is not a regular file (possibly a pipe?).
98565 Unspecified internal error.
108925 Error opening <_io.BytesIO object at 0x00000271EE38B290>: File does not exist or is not a regular file (possibly a pipe?).
133297 Error opening <_io.BytesIO object at 0x00000271EE38B290>: File does not exist or is not a regular file (possibly a pipe?).
16820 System error.
107248 System error.
132589 System error.
40659 System error.
146481 System error.
97283 System error.
66637 System error.
88959 System error.
72149 System error.
121737 System error.
68354 System error.
694 System error.
119027 System error.
144214 System error.
143307 System error.
54703 System error.
47193 System error.
75377 System error.
99501 System error.
37131 System error.
91894 System error.
56639 System error.
7011 System error.
133436 System error.
128760 System error.
7393 System error.
38894 System error.
91443 System error.
97887 System

In [ ]:
import io
import os
import numpy as np
from pydub import AudioSegment
import soundfile as sf


for df, out_dir in zip(
    [df_train, df_val],
    ["data/audio_train", "data/audio_val"]
):

    os.makedirs(out_dir, exist_ok=True)

    for _, row in df.iterrows():

        track_id = row["track_id"]
        out_path = f"{out_dir}/{track_id}.wav"

        if os.path.exists(out_path):
            continue

        try:
            audio_bytes = row["audio"]["bytes"]


            audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")

            samples = np.array(audio.get_array_of_samples())

            if audio.channels > 1:
                samples = samples.reshape((-1, audio.channels)).mean(axis=1)

            samples = samples.astype(np.float32)
            samples /= (np.max(np.abs(samples)) + 1e-9)

            sf.write(out_path, samples, audio.frame_rate)

        except Exception as e:
            print(track_id, e)

In [68]:

path = "data/audio_val"

num_files = sum(
    len(files)
    for _, _, files in os.walk(path)
)

print(f"Количество файлов: {num_files}")

Количество файлов: 1600


In [ ]:
# # Нормализация VS Без нормализации

# import numpy as np
# from PIL import Image

# k = 0

# for filename in os.listdir("data/audio_train"):
    
#     track_id = filename.replace(".wav", "")
#     y, sr = librosa.load(f"data/audio_train/{filename}", sr=None)
    
#     mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
#     mel_db = librosa.power_to_db(mel, ref=np.max)
    
#     mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min()) * 255
#     img = Image.fromarray(mel_norm.astype(np.uint8))
#     img.save(f"data/test/255_{track_id}.png")
    
#     k+=1
#     if k == 10: 
#         k = 0
#         break
        

# for filename in os.listdir("data/audio_train"):
#     track_id = filename.replace(".wav", "")
#     y, sr = librosa.load(f"data/audio_train/{filename}", sr=None)

#     mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
#     mel_db = librosa.power_to_db(mel, ref=np.max)

#     mel_norm = (mel_db + 80) / 80 * 255

#     img = Image.fromarray(mel_norm.astype(np.uint8))
#     img.save(f"data/test/{track_id}.png")

#     k += 1
#     if k == 10:
#         break

In [ ]:
import numpy as np
from PIL import Image


for filename in os.listdir("data/audio_train"):
    track_id = filename.replace(".wav", "")
    y, sr = librosa.load(f"data/audio_train/{filename}", sr=None)
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min()) * 255
    # (mel_db + 80) / 80 * 255 
    # 255 - яркость пикселя

    img = Image.fromarray(mel_norm.astype(np.uint8))
    img.save(f"data/melspecs_train/{track_id}.png")

for filename in os.listdir("data/audio_val"):
    track_id = filename.replace(".wav", "")
    y, sr = librosa.load(f"data/audio_val/{filename}", sr=None)
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min()) * 255
    img = Image.fromarray(mel_norm.astype(np.uint8))
    img.save(f"data/melspecs_val/{track_id}.png")

C:\Users\tim7m\AppData\Local\Temp\ipykernel_1191684\1939384772.py:12: RuntimeWarning: invalid value encountered in divide
  mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min()) * 255
C:\Users\tim7m\AppData\Local\Temp\ipykernel_1191684\1939384772.py:13: RuntimeWarning: invalid value encountered in cast
  img = Image.fromarray(mel_norm.astype(np.uint8))
C:\Users\tim7m\AppData\Local\Temp\ipykernel_1191684\1939384772.py:23: RuntimeWarning: invalid value encountered in divide
  mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min()) * 255
C:\Users\tim7m\AppData\Local\Temp\ipykernel_1191684\1939384772.py:24: RuntimeWarning: invalid value encountered in cast
  img = Image.fromarray(mel_norm.astype(np.uint8))


In [67]:
import io
import soundfile as sf

mono = 0
stereo = 0
errors = 0

for _, row in df_train.iterrows():
    try:
        audio_bytes = row["audio"]["bytes"]

        y, sr = sf.read(io.BytesIO(audio_bytes))

        if len(y.shape) == 1:
            mono += 1
        else:
            stereo += 1

    except:
        errors += 1

print("mono:", mono)
print("stereo:", stereo)
print("errors:", errors)

mono: 71
stereo: 6325
errors: 4


In [32]:
import os
from PIL import Image
import numpy as np
from collections import Counter

folders = ["data/melspecs_train", "data/melspecs_val"]

for folder in folders:
    print(f"\n=== {folder} ===")
    sizes = Counter()
    empty_or_bad = []

    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        try:
            img = Image.open(path)
            sizes[img.size] += 1

            arr = np.array(img)
            if arr.max() == arr.min():  # вся картинка одного цвета = пустая/битая
                empty_or_bad.append(fname)

        except Exception as e:
            print("Ошибка открытия:", fname, e)
            empty_or_bad.append(fname)

    print("Всего файлов:", sum(sizes.values()))
    print("Размеры (топ-10):")
    for size, count in sizes.most_common(10):
        print(f"  {size}: {count}")

    print(f"Подозрительных (пустых/однотонных) файлов: {len(empty_or_bad)}")
    if empty_or_bad:
        print("Примеры:", empty_or_bad[:10])


=== data/melspecs_train ===


KeyboardInterrupt: 

In [65]:
import os

for folder in ["data/audio_train", "data/audio_val"]:
    sizes = []
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        sizes.append(os.path.getsize(path))
    
    sizes_arr = sorted(sizes)
    print(folder)
    print("  мин:", sizes_arr[0], "медиана:", sizes_arr[len(sizes_arr)//2], "макс:", sizes_arr[-1])
    print("  файлов <1000 байт:", sum(1 for s in sizes if s < 1000))

data/audio_train
  мин: 24620 медиана: 5287912 макс: 5757928
  файлов <1000 байт: 0
data/audio_val
  мин: 24620 медиана: 2646282 макс: 5757928
  файлов <1000 байт: 0


In [61]:
import os

bad_wav_train = []
for f in os.listdir("data/audio_train"):
    path = f"data/audio_train/{f}"
    if os.path.getsize(path) < 10000:  # пустой/почти пустой wav
        bad_wav_train.append(f)
        os.remove(path)
        track_id = f.replace(".wav", "")
        png_path = f"data/melspecs_train/{track_id}.png"
        if os.path.exists(png_path):
            os.remove(png_path)

print("Удалено битых train:", len(bad_wav_train))

# то же для val
bad_wav_val = []
for f in os.listdir("data/audio_val"):
    path = f"data/audio_val/{f}"
    if os.path.getsize(path) < 10000:
        bad_wav_val.append(f)
        os.remove(path)
        track_id = f.replace(".wav", "")
        png_path = f"data/melspecs_val/{track_id}.png"
        if os.path.exists(png_path):
            os.remove(png_path)

print("Удалено битых val:", len(bad_wav_val))

Удалено битых train: 1375
Удалено битых val: 473


In [88]:
import io
import librosa
import numpy as np

mels = []
labels = []
track_ids = []

for i, row in df_train.iterrows():

    try:
        audio_bytes = row["audio"]["bytes"]

        y, sr = librosa.load(io.BytesIO(audio_bytes))

        mel = librosa.feature.melspectrogram(y=y, sr=sr)

        mel_db = librosa.power_to_db(mel, ref=np.max)

        mel_norm = (mel_db + 80) / 80

        mels.append(mel_norm.astype(np.float32))
        labels.append(row["genre"])
        track_ids.append(row["track_id"])

    except Exception as e:
        print(row["track_id"], e)

99134 Error opening <_io.BytesIO object at 0x00000272205F5FD0>: File does not exist or is not a regular file (possibly a pipe?).
98565 Unspecified internal error.
108925 Error opening <_io.BytesIO object at 0x00000271EE370D10>: File does not exist or is not a regular file (possibly a pipe?).
133297 Error opening <_io.BytesIO object at 0x00000271EE370D10>: File does not exist or is not a regular file (possibly a pipe?).
10668 Unable to allocate 10.1 MiB for an array with shape (1025, 1291) and data type complex64
52039 Unable to allocate 646. KiB for an array with shape (661560,) and data type bool
62525 Unable to allocate 10.1 MiB for an array with shape (1321967, 2) and data type float32
10992 Unable to allocate 10.1 MiB for an array with shape (1323119, 2) and data type float32
126402 Unable to allocate 10.1 MiB for an array with shape (1323119, 2) and data type float32
122934 Unable to allocate 2.52 MiB for an array with shape (660984,) and data type float32
145058 Unable to allocat

In [94]:
from collections import Counter

lengths = [mel.shape[1] for mel in mels]

print(Counter(lengths))

Counter({1291: 3387, 1293: 2668, 1292: 324})


In [95]:
mels = [mel[:, :1291] for mel in mels]



In [98]:
np.savez_compressed(
    "melspec.npz",
    mels=np.array(mels, dtype=np.float16),
    labels=np.array(labels),
    track_ids=np.array(track_ids)
)

MemoryError: Unable to allocate 1.96 GiB for an array with shape (6379, 128, 1291) and data type float16

In [100]:
import os
import numpy as np

os.makedirs("data/melsnpy", exist_ok=True)

for mel, track_id in zip(mels, track_ids):
    np.save(
        f"data/melsnpy/{track_id}.npy",
        mel.astype(np.float16)
    )

In [101]:
import pandas as pd

pd.DataFrame({
    "track_id": track_ids,
    "label": labels
}).to_csv("labels.csv", index=False)

In [102]:
np.savez_compressed(
    "melspec.npz",
    mels=np.array(mels, dtype=np.float16),
    labels=np.array(labels),
    track_ids=np.array(track_ids)
)